## Import

In [1]:
import glob
import copy
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


from create_df import create_df
from train_logistic_regression import train_logistic_regression

## Main

### Load, Split, Train, Test

In [ ]:

def get_csv_file_path():
	# If running from root
	csv_folder_root = "data/loose_csv/*.csv"
	path_a = glob.glob(csv_folder_root)
	
	# If running from inside src
	csv_folder_src = "../data/loose_csv/*.csv"
	path_b = glob.glob(csv_folder_src)

	# Assign file_paths
	# file_paths = None
	if(len(path_a) > len(path_b)):
		return path_a
	return path_b


class SplitData:
	def __init__(self, x_train, x_test, y_train, y_test):
		self.x_train = x_train
		self.x_test = x_test
		self.y_train = y_train
		self.y_test = y_test



print("Starting Main")

# ==== Load Data ==== 
print("Getting Paths")
# --- Finding CSV Files ---
file_paths = get_csv_file_path()

# --- Form Date Frame ---
print("Creating Dataframe")
df = create_df(file_paths)

# print("Dataframe Created:",df.shape)


# === Spliting Data ===
print("Splitting Data")

# --- Input ---
x = df.drop(
	columns = [
		"Attack Name", 
		"Label",
		"Flow ID",
		"Src IP",
		"Dst IP",
		"Timestamp"
	]
)
# print(x.head())

	
'''Logistic Regression doesnt like Strinsg, so in addition
to dropping output values, we need to drop strings
IP values may apply to these collected pieces of data,
but in general they might not so should be removed. Same
with time.'''
# print(x.select_dtypes(exclude="number").columns.tolist())

# --- Output ---
y = df["Attack Name"]
# print(y.head())

x_train, x_test, y_train, y_test = train_test_split(
	x,
	y,
	test_size=.2,
	random_state=42,
	stratify=y
)

# --- Standardization of X ---
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)


# === Train Models on Specific Algorithms ===
print("Training Model")
model_lr = train_logistic_regression(x_train_scaled, y_train, 300)

print("Model Trained")

# === Test Model ===
print("Testing Model")
model_predictions = model_lr.predict(x_test_scaled)

print("Model Tested")

# === Validate Model Predictions ===
# results = DataValidation(y_test, model_predictions)
# print(results.get_all_attack_accuracies())

# validation.get_all_attack_accuracies()


Starting Main
Getting Paths
Creating Dataframe
Splitting Data
Training Model


c:\Projects\a.School\MachineLearning\a.FinalProject\NetworkTrafficClassifier\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model Trained
Testing Model


## Validate

### Validate All Results

In [ ]:
def validate_all(y_test, model_predictions):

	# --- Count Correct Predictions ---
	correctPredictionCount = np.sum(y_test == model_predictions)

	# --- Calculate Percent Correct ---
	accuracy = correctPredictionCount / model_predictions.size
	
	print("===========")
	print("Accuracy:",accuracy)
	print("===========")
	print("Y Test:\n",y_test)
	print("===========")
	print("Model Predictions:\n", model_predictions)
	print("===========")


validate_all(y_test, model_predictions)

Accuracy: 0.9635609830307703
Y Test:
 3032205    Recon Port Scan
1586384      DoS TCP Flood
1539971      DoS TCP Flood
3046747    Recon Port Scan
236918       DoS TCP Flood
                ...       
3360475    Recon Port Scan
1143621      DoS TCP Flood
833940       DoS TCP Flood
1791706      DoS TCP Flood
3363232    Recon Port Scan
Name: Attack Name, Length: 677049, dtype: str
Model Predictions:
 ['Recon Port Scan' 'DoS TCP Flood' 'DoS TCP Flood' ... 'DoS TCP Flood'
 'DoS TCP Flood' 'Recon Port Scan']


### Validate Benign Traffic

In [51]:
# List of Attack Types
attack_types = [
	"Benign Traffic",
	"DDoS ICMP Flood",
	"DDoS UDP Flood",
	"DoS ICMP Flood",
	"DoS TCP Flood",
	"DoS UDP Flood",
	"MITM ARP Spoofing",
	"MQTT DDoS Publish Flood",
	"MQTT DoS Connect Flood",
	"MQTT DoS Publish Flood",
	"MQTT Malformed",
	"MQTT Malformed",
	"Recon OS Scan",
	"Recon Port Scan",
	"Recon Vulnerability Scan"
]

# Class to validate and store accuracy
class DataValidation:
	# Constructor
	def __init__(self, y_test, model_predictions):
		self.y_test = np.asarray(y_test)
		self.model_predictions = np.asarray(model_predictions)
		if self.y_test.size != self.model_predictions.size:
			raise ValueError("y_test and model_predictions must have the same length")

		self.accuracy_by_attack = {}
		self.overall_accuracy = None

	# Returns indicies in y_test of specified attack types
	def get_attack_indices(self, attack_type):
		if attack_type not in attack_types:
			raise ValueError(f"Unknown attack type: {attack_type}")
		indices = np.flatnonzero(self.y_test == attack_type).tolist()
		if not indices:
			raise ValueError(f"Attack type is not present in y_test: {attack_type}")
		return indices

	# Returns Model Predictions at speficied indicies
	def get_prediction_subset(self, indices):
		if any(index < 0 or index >= self.model_predictions.size for index in indices):
			raise IndexError("Prediction index is outside the test data")
		return self.model_predictions[indices].tolist()

	# Calculates and Returns the accuracy for a speified indicie
	def validate_attack(self, attack_type, indices=None):
		if indices is None:
			indices = self.get_attack_indices(attack_type)
		predictions = self.get_prediction_subset(indices)
		correct_count = sum(prediction == attack_type for prediction in predictions)
		accuracy = correct_count / len(indices)
		self.accuracy_by_attack[attack_type] = accuracy
		return accuracy

	# Calculate and store accuracy across the complete test dataset.
	def validate_all(self):
		correct_count = np.sum(self.y_test == self.model_predictions)
		self.overall_accuracy = correct_count / self.model_predictions.size
		return self.overall_accuracy

	# Calculate every per-attack score and the overall score once.
	def run_validation(self):
		for attack_type in dict.fromkeys(attack_types):
			indices = self.get_attack_indices(attack_type)
			self.validate_attack(attack_type, indices)
		self.validate_all()
		return self

	# === Methods to get Accuracies ===
	# Returns the accuracy for a specified attack type
	def get_attack_accuracy(self, attack_type):
		return self.accuracy_by_attack[attack_type]

	# Returns a dictionary containing all attack types and corisponding accuracy
	def get_all_attack_accuracies(self):
		return self.accuracy_by_attack.copy()

	# Returns model accuracy
	def get_overall_accuracy(self):
		return self.overall_accuracy


validation = DataValidation(y_test, model_predictions).run_validation()
print("Overall Acuracy:",validation.get_overall_accuracy())
print("All Accuracies:",validation.get_all_attack_accuracies())


Overall Acuracy: 0.9635609830307703
All Accuracies: {'Benign Traffic': 0.9359288779889638, 'DDoS ICMP Flood': 0.060784313725490195, 'DDoS UDP Flood': 0.02912621359223301, 'DoS ICMP Flood': 0.0166270783847981, 'DoS TCP Flood': 0.9991100753706833, 'DoS UDP Flood': 0.09149277688603531, 'MITM ARP Spoofing': 0.037914691943127965, 'MQTT DDoS Publish Flood': 0.9809381153135306, 'MQTT DoS Connect Flood': 0.9819350502037558, 'MQTT DoS Publish Flood': 0.612565445026178, 'MQTT Malformed': 0.017817371937639197, 'Recon OS Scan': 0.00785324972161988, 'Recon Port Scan': 0.9981772308326039, 'Recon Vulnerability Scan': 0.004206730769230769}


### Using `DataValidation`

The class is created and run with the test labels and model predictions:

```python
validation = DataValidation(y_test, model_predictions).run_validation()
```

The full validation run calculates each result once and stores the results on the object:

```python
# Get the accuracy for one attack type.
validation.get_attack_accuracy("DoS TCP Flood")

# Get a dictionary containing every attack type and its accuracy.
validation.get_all_attack_accuracies()

# Get the accuracy across the complete test dataset.
validation.get_overall_accuracy()
```

The intermediate methods can also be used when inspecting one attack type:

```python
indices = validation.get_attack_indices("DoS TCP Flood")
predictions = validation.get_prediction_subset(indices)
```

`indices` contains the positions where that attack appears in `y_test`, and `predictions` contains the model predictions at those same positions.
